In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 1: LOGGING AND IMPORTS - MODIFY THESE PATHS AS NEEDED
# ═══════════════════════════════════════════════════════════════════════════════
#jupyter cell test notebook for pipeline integration 
# DO NOT OVERWRITE LOGS, CREATE NEW LOG FILE BEFORE EACH NEW RUN WITH TIME AND DATE 
import os
import re
import io
import sys
import json
import ujson
import ast
import time
import glob
from pathlib import Path
import logging

# ═══════════════════════════════════════════════════════════════════════════════
# AUTOMATIC PATH SETUP - NO NEED TO MODIFY UNLESS YOU WANT CUSTOM PATHS
# ═══════════════════════════════════════════════════════════════════════════════

# Get the directory where this notebook is located
NOTEBOOK_DIR = Path.cwd()

# Set up paths relative to notebook location
LOG_DIR = NOTEBOOK_DIR / "logs"
PDF_INPUT_FOLDER = NOTEBOOK_DIR / "sample_pdfs"
OUTPUT_FOLDER = NOTEBOOK_DIR / "output"

# Create directories if they don't exist
LOG_DIR.mkdir(exist_ok=True)
PDF_INPUT_FOLDER.mkdir(exist_ok=True)
OUTPUT_FOLDER.mkdir(exist_ok=True)

# Convert to strings for compatibility
LOG_DIR = str(LOG_DIR)
PDF_INPUT_FOLDER = str(PDF_INPUT_FOLDER)
OUTPUT_FEATHER = str(OUTPUT_FOLDER / "simple_docling_results.feather")

LOG_FILE = os.path.join(LOG_DIR, "docling_testing.log")
logging.basicConfig(
    filename=LOG_FILE,
    filemode='w',
    level=logging.DEBUG,
    format='%(asctime)s - %(levelname)s - %(message)s'
)

# set loggers for docling modules
docling_logger = logging.getLogger("docling")
docling_logger.setLevel(logging.DEBUG)

# Ensure logs propagate to the root logger
docling_logger.propagate = True
_log = logging.getLogger(__name__) 

_log.debug("Test debug message from LayoutPostprocessor.py")
_log.info("Test info message from LayoutPostprocessor.py")

def print_and_log(message, level="info"):
    """Print to console and log at the same time"""
    print(message)
    if level == "info":
        _log.info(message)
    elif level == "debug":
        _log.debug(message)
    elif level == "error":
        _log.error(message)
    elif level == "warning":
        _log.warning(message)

import traceback  # [QA CHANGE] For logging full tracebacks
import requests
import pandas as pd

import tiktoken
from concurrent.futures import ThreadPoolExecutor, as_completed
from threading import Lock

print("="*80)
print("SIMPLE PDF PROCESSOR SETUP")
print("="*80)
print(f"📁 Put your PDF files in: {PDF_INPUT_FOLDER}")
print(f"📄 Results will be saved to: {OUTPUT_FEATHER}")
print(f"📋 Logs will be saved to: {LOG_DIR}")
print("="*80)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 2: CONFIGURATION - PATHS ARE NOW AUTOMATIC
# ═══════════════════════════════════════════════════════════════════════════════

print_and_log(f"PDF Input Folder: {PDF_INPUT_FOLDER}")
print_and_log(f"Output File: {OUTPUT_FEATHER}")
print_and_log(f"Log Directory: {LOG_DIR}")


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 3: IMPORT DOCLING PROCESSING FUNCTION
# ═══════════════════════════════════════════════════════════════════════════════

try:
    from docling_test_single_GPU import do_docling_extraction
    print_and_log("[+] Successfully imported docling extraction function")
except ImportError as e:
    print_and_log(f"[!] Could not import docling extraction function: {e}", "error")
    print_and_log("    Make sure 'docling_test_single_GPU.py' is in the same directory as this notebook", "error")


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 4: PDF DISCOVERY FUNCTION
# ═══════════════════════════════════════════════════════════════════════════════

def create_pdf_dataframe(pdf_folder):
    """
    Scan a folder for PDF files and create a minimal DataFrame for processing.
    Args:
        pdf_folder (str): Path to folder containing PDF files
    Returns:
        pd.DataFrame: DataFrame with PDFPath column
    """
    # Check if folder exists
    if not os.path.exists(pdf_folder):
        raise FileNotFoundError(f"PDF folder not found: {pdf_folder}")
    
    # Find all PDF files in the folder
    pdf_pattern = os.path.join(pdf_folder, "*.pdf")
    pdf_files = glob.glob(pdf_pattern)
    
    if not pdf_files:
        raise ValueError(f"No PDF files found in {pdf_folder}")
    
    print_and_log(f"[+] Found {len(pdf_files)} PDF files:")
    for pdf_file in pdf_files:
        print_and_log(f"    - {os.path.basename(pdf_file)}")
    
    # Create DataFrame - just PDFPath column
    df = pd.DataFrame({
        "PDFPath": pdf_files,
        "FileName": [os.path.basename(path) for path in pdf_files]
    })
    
    print_and_log(f"Created DataFrame with {len(df)} PDF files")
    
    return df

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 5: CREATE DATAFRAME FROM YOUR PDF FOLDER
# ═══════════════════════════════════════════════════════════════════════════════

print_and_log(f"\nScanning for PDFs in: {PDF_INPUT_FOLDER}")

# Check if input folder exists
if not os.path.exists(PDF_INPUT_FOLDER):
    print_and_log(f"[!] Input folder does not exist: {PDF_INPUT_FOLDER}", "error")
    print_and_log("Please add PDF files to the 'sample_pdfs' folder", "error")
else:
    try:
        pdf_df = create_pdf_dataframe(PDF_INPUT_FOLDER)
        print_and_log(f"[+] Created DataFrame with {len(pdf_df)} files")
        
        # Display the DataFrame
        print_and_log("\nDataFrame contents:")
        print(pdf_df)
    except ValueError as e:
        print_and_log(f"[!] {e}", "error")
        print_and_log("Please add some PDF files to the 'sample_pdfs' folder", "error")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 6: PROCESS PDFs WITH DOCLING
# ═══════════════════════════════════════════════════════════════════════════════

if 'pdf_df' in locals() and len(pdf_df) > 0:
    print_and_log(f"Processing {len(pdf_df)} PDFs with Docling...")
    print_and_log("This may take several minutes depending on PDF size and complexity...")
    
    try:
        processed_df = do_docling_extraction(pdf_df)
        print_and_log(f"[+] Successfully processed {len(processed_df)} files")
        print_and_log(f"[+] DataFrame now has {len(processed_df.columns)} columns")
        print_and_log(f"Columns: {list(processed_df.columns)}")
        
    except Exception as e:
        print_and_log(f"[!] Error during processing: {e}", "error")
else:
    print_and_log("[!] No PDFs to process. Please add PDF files to the 'sample_pdfs' folder.", "error")


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 7: SAVE RESULTS
# ═══════════════════════════════════════════════════════════════════════════════

if 'processed_df' in locals():
    try:
        processed_df.to_feather(OUTPUT_FEATHER)
        print_and_log(f"[+] Results saved to: {OUTPUT_FEATHER}")
        
        # Display summary
        print_and_log(f"\nProcessing Summary:")
        print_and_log(f"  - Input PDFs: {len(pdf_df)}")
        print_and_log(f"  - Successfully processed: {len(processed_df)}")
        print_and_log(f"  - Output columns: {len(processed_df.columns)}")
        print_and_log(f"  - Log file: {LOG_FILE}")
        
    except Exception as e:
        print_and_log(f"[!] Error saving results: {e}", "error")
else:
    print_and_log("[!] No processed data to save.")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 8: INSPECT RESULTS
# ═══════════════════════════════════════════════════════════════════════════════

def inspect_results(feather_path=OUTPUT_FEATHER):
    """Load and display the processing results."""
    if not os.path.exists(feather_path):
        print_and_log(f"[!] Results file not found: {feather_path}")
        return None
        
    try:
        df = pd.read_feather(feather_path)
        print_and_log(f"[+] Loaded results: {len(df)} rows, {len(df.columns)} columns")
        print_and_log(f"\nColumns: {list(df.columns)}")
        print_and_log(f"\nFirst few rows:")
        display(df.head())
        return df
        
    except Exception as e:
        print_and_log(f"[!] Error loading results: {e}", "error")
        return None

# Run inspection if results exist
if os.path.exists(OUTPUT_FEATHER):
    results_df = inspect_results()

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 9: UTILITY - TEST SINGLE PDF
# ═══════════════════════════════════════════════════════════════════════════════

def test_single_pdf(pdf_path):
    """Test processing on a single PDF file."""
    if not os.path.exists(pdf_path):
        print_and_log(f"[!] PDF file not found: {pdf_path}")
        return
        
    print_and_log(f"[+] Testing single PDF: {os.path.basename(pdf_path)}")
    
    # Create single-row DataFrame
    test_df = pd.DataFrame({
        "PDFPath": [pdf_path],
        "FileName": [os.path.basename(pdf_path)]
    })
    
    try:
        result_df = do_docling_extraction(test_df)
        print_and_log(f"[+] Successfully processed!")
        print_and_log(f"Result columns: {list(result_df.columns)}")
        display(result_df.head())
        return result_df
        
    except Exception as e:
        print_and_log(f"[!] Error processing PDF: {e}", "error")
        return None

# Example usage (uncomment to test a specific file):
# test_single_pdf(os.path.join(PDF_INPUT_FOLDER, "your_pdf_file.pdf"))

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 10: QUICK PROCESSING FUNCTION (ALTERNATIVE TO RUNNING ALL CELLS)
# ═══════════════════════════════════════════════════════════════════════════════

def quick_process_folder(folder_path=None, output_path=None):
    """
    Quick function to process all PDFs in a folder.
    Args:
        folder_path (str, optional): Path to folder containing PDFs (defaults to sample_pdfs)
        output_path (str, optional): Output file path (defaults to output folder)
    """
    if folder_path is None:
        folder_path = PDF_INPUT_FOLDER
    
    if output_path is None:
        output_path = OUTPUT_FEATHER
    
    print_and_log(f"Processing PDFs from: {folder_path}")
    
    # Create DataFrame
    df = create_pdf_dataframe(folder_path)
    
    # Process with Docling
    processed_df = do_docling_extraction(df)
    
    # Save results
    processed_df.to_feather(output_path)
    
    print_and_log(f"Results saved to: {output_path}")
    return processed_df

# Example usage:
# results = quick_process_folder()